# 04 — Delivery Time Regression Model
**Week 3, Steps 6–9 (Day 13)**

Builds the delivery time prediction model from the ABT.  
Features = everything known at order placement. Target = `delivery_time_minutes`.

> Run **Restart & Run All** to reproduce.


In [1]:
import sys
from pathlib import Path
if Path.cwd().name == "notebooks": import os; os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, json, sklearn

from sklearn.model_selection import train_test_split, KFold

from src.model_delivery import (
    NUMERIC, CATEGORICAL, EXCLUDED, TARGET, KPI_R2, KPI_RMSE,
    build_model_dataset, make_preprocessor, MODELS,
    compare_models, tune_model, evaluate,
    plot_residuals, plot_feature_importance,
)

rng = np.random.default_rng(42)
pd.set_option("display.float_format", "{:.4f}".format)
print(f"scikit-learn version: {sklearn.__version__}")
print("Libraries loaded ✅")


scikit-learn version: 1.9.1
Libraries loaded ✅


## 1. Build the modelling dataset (Step 6)

In [2]:
abt = pd.read_csv("data/cleaned/analytical_base_table.csv")
abt["order_date"] = pd.to_datetime(abt["order_date"], errors="coerce")
for col in ["delivery_time_minutes","food_cost","final_amount","order_hour","dt_outlier_flag"]:
    abt[col] = pd.to_numeric(abt[col], errors="coerce")

X, y = build_model_dataset(abt)
print(f"Rows: {len(X):,}")
print(f"Numeric features:     {len([c for c in NUMERIC     if c in X.columns])}")
print(f"Categorical features: {len([c for c in CATEGORICAL if c in X.columns])}")
print(f"Target — mean: {y.mean():.1f} min  std: {y.std():.1f} min")
print()
print("Excluded columns (with reasons):")
for col, reason in EXCLUDED.items():
    print(f"  {col:<40} {reason}")


Rows: 11,193
Numeric features:     23
Categorical features: 6
Target — mean: 33.5 min  std: 9.9 min

Excluded columns (with reasons):
  feedback_customer_rating                 post-delivery — unknown at order time
  feedback_delivery_rating                 post-delivery — unknown at order time
  feedback_food_rating                     post-delivery — unknown at order time
  average_rating                           post-delivery — unknown at order time
  sentiment                                post-delivery — unknown at order time
  review                                   post-delivery — free text
  order_status                             every row is 'Delivered'; zero variance
  order_id                                 identifier — not a predictor
  customer_id                              identifier — use customer attributes instead
  restaurant_id                            identifier — use restaurant attributes instead
  delivery_partner_id                      identifier — use

## 2. Train/test split (Step 7)

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
cv = KFold(n_splits=5, shuffle=True, random_state=42)
print(f"Train: {len(X_train):,}  Test: {len(X_test):,}")
print("Note: split BEFORE any preprocessing — preprocessing fitted only on training data")


Train: 8,954  Test: 2,239
Note: split BEFORE any preprocessing — preprocessing fitted only on training data


## 3. Cross-validate all models (Step 7)

In [4]:
print("Cross-validating 5 models on training set...")
cv_table = compare_models(X_train, y_train, cv)
cv_table


Cross-validating 5 models on training set...


    Baseline (predict mean)             RMSE=9.94  R²=-0.0003


    Ridge Regression                    RMSE=10.01  R²=-0.0130


    Decision Tree (depth=8)             RMSE=10.34  R²=-0.0813


    Random Forest                       RMSE=10.04  R²=-0.0191


    Gradient Boosting                   RMSE=10.18  R²=-0.0482


,model,cv_mae,cv_rmse,cv_rmse_std,cv_r2
0,Baseline (predict mean),7.9790,9.9440,0.0670,-0.0003
1,Ridge Regression,8.0190,10.0070,0.0690,-0.0130
2,Random Forest,8.0460,10.0370,0.0510,-0.0191
3,Gradient Boosting,8.1550,10.1790,0.1230,-0.0482
4,Decision Tree (depth=8),8.2350,10.3380,0.0540,-0.0813


In [5]:
# Compare vs baseline and KPI targets
print(f"Baseline CV RMSE: {cv_table.loc[cv_table['model'].str.contains('Baseline'),'cv_rmse'].iloc[0]:.3f}")
print(f"Best model CV RMSE: {cv_table.iloc[0]['cv_rmse']:.3f}  (model: {cv_table.iloc[0]['model']})")
print(f"KPI target RMSE: ≤{KPI_RMSE} min  — {'⚠️ not met — see honest assessment' if cv_table.iloc[0]['cv_rmse'] > KPI_RMSE else '✅ met'}")
print()
print("Honest assessment:")
print("  All models perform at baseline level (R² ≈ 0).")
print("  Root cause: delivery_time_minutes has near-zero feature correlations")
print("  (max |r| = 0.047) in this dataset.")
print("  The pipeline is correct — this is a data property, not a code error.")


Baseline CV RMSE: 9.944
Best model CV RMSE: 9.944  (model: Baseline (predict mean))
KPI target RMSE: ≤6.0 min  — ⚠️ not met — see honest assessment

Honest assessment:
  All models perform at baseline level (R² ≈ 0).
  Root cause: delivery_time_minutes has near-zero feature correlations
  (max |r| = 0.047) in this dataset.
  The pipeline is correct — this is a data property, not a code error.


## 4. Load pre-tuned model and evaluate on test set (Step 9)

In [6]:
best_model = joblib.load("models/delivery_time_best_model.pkl")
test_metrics, pred = evaluate(best_model, X_test, y_test)

print("=== Test set results (opened ONCE) ===")
for k, v in test_metrics.items():
    print(f"  {k:<10} {v}")

print()
r2_ok   = test_metrics['R2']   >= KPI_R2
rmse_ok = test_metrics['RMSE'] <= KPI_RMSE
print(f"R² KPI (≥{KPI_R2}):       {'✅ PASS' if r2_ok   else '⚠️ below target'}")
print(f"RMSE KPI (≤{KPI_RMSE} min): {'✅ PASS' if rmse_ok else '⚠️ below target'}")


=== Test set results (opened ONCE) ===
  MAE        7.84
  MSE        95.532
  RMSE       9.774
  R2         -0.0022

R² KPI (≥0.75):       ⚠️ below target
RMSE KPI (≤6.0 min): ⚠️ below target


## 5. Residual analysis (Step 9)

In [7]:
plot_residuals(y_test, pred, "Random Forest")


  → images/delivery_residuals.png saved


## 6. Feature importance (Step 9)

In [8]:
from src.model_delivery import save_prediction_export, interpretation_section

# Power BI export + permutation importance (chart and CSV) + computed interpretation
save_prediction_export(abt, X_test, y_test, pred)
imp_df = plot_feature_importance(best_model, X_test, y_test, "Random Forest")
print(imp_df.head(10).to_string(index=False))
print()
print(interpretation_section(test_metrics))

  → data/cleaned/ml_delivery_predictions.csv saved


  → images/delivery_feature_importance.png + ml_delivery_feature_importance.csv saved
                   feature  importance    std                                         direction distinguishable_from_noise
            payment_method      0.0050 0.0016                 categorical - no single direction                        yes
restaurant_popularity_rank      0.0048 0.0015              no meaningful relationship (r=0.008)                        yes
              basket_value      0.0037 0.0024              no meaningful relationship (r=0.008)                         no
           restaurant_city      0.0037 0.0020                 categorical - no single direction                         no
 partner_avg_delivery_time      0.0034 0.0013 higher values go with shorter delivery (r=-0.021)                        yes
        restaurant_cuisine      0.0032 0.0020                 categorical - no single direction                         no
                  humidity      0.0027 0.0016        

## 7. Error analysis — where is the model worst?

In [9]:
errors_df = X_test.copy()
errors_df["abs_error"] = np.abs(y_test.values - pred)
errors_df["actual"]    = y_test.values
errors_df["predicted"] = pred

print("Mean absolute error by peak_hour:")
print(errors_df.groupby("peak_hour")["abs_error"].mean().round(2))

print()
print("Mean absolute error by restaurant_city (top 5 worst):")
print(errors_df.groupby("restaurant_city")["abs_error"].mean()
      .sort_values(ascending=False).head(5).round(2))

print()
print("Mean absolute error by vehicle_type:")
print(errors_df.groupby("vehicle_type")["abs_error"].mean()
      .sort_values(ascending=False).round(2))


Mean absolute error by peak_hour:
peak_hour
0   7.7900
1   7.9200
Name: abs_error, dtype: float64

Mean absolute error by restaurant_city (top 5 worst):
restaurant_city
Coimbatore   9.6100
Bengaluru    9.0300
Indore       8.6200
Bhopal       8.5900
Nashik       8.5100
Name: abs_error, dtype: float64

Mean absolute error by vehicle_type:
vehicle_type
Bike      8.0400
Scooter   7.7300
Bicycle   7.6500
Car       7.1900
Name: abs_error, dtype: float64


## 8. Model metadata

In [10]:
with open("models/model_metadata.json") as f:
    meta = json.load(f)
print(json.dumps(meta["delivery_model"], indent=2))


{
  "target": "delivery_time_minutes",
  "model": "Random Forest",
  "model_file": "delivery_time_best_model.pkl",
  "sklearn_version": "1.9.1",
  "trained_on": "2026-11-07",
  "test_metrics": {
    "MAE": 7.84,
    "MSE": 95.532,
    "RMSE": 9.774,
    "R2": -0.0022
  },
  "kpi_r2_target": 0.75,
  "kpi_rmse_target_min": 6.0,
  "features": {
    "numeric": [
      "order_hour",
      "order_dayofweek",
      "peak_hour",
      "weekend_order",
      "n_items",
      "avg_prep_time",
      "max_prep_time",
      "basket_value",
      "discount_pct",
      "coupon_used",
      "restaurant_rating",
      "restaurant_popularity_rank",
      "partner_rating",
      "delivery_efficiency",
      "partner_avg_delivery_time",
      "completed_deliveries",
      "is_cross_city",
      "customer_tenure_days",
      "rainfall",
      "temperature",
      "humidity",
      "traffic_score",
      "is_peak_and_rain"
    ],
    "categorical": [
      "restaurant_city",
      "restaurant_cuisine",
    